In [26]:
import cadquery as cq

# UNITS ARE MM

###############################################################################
# PARAMETERS
###############################################################################
R                           = 50
H                           = 500
fillet_r                    = 30
shell_thickness             = 5

inner_fluid_r               = 30
inner_fluid_depth           = 50

outer_fluid_r               = 15
outer_fluid_depth           = 50
outer_fluid_inlet_z         = H * 0.8
outer_fluid_outlet_z        = H * 0.2

baffle_radius               = R
baffle_thickness            = 10
baffle_start                = fillet_r + H / 20
baffle_end                  = H - H / 20 - fillet_r

hole_diameter               = 15
col_spacing                 = baffle_radius / 2
row_spacing                 = baffle_radius / 2
holes_per_row               = [3, 4, 3]

num_semi_baffles            = 4
pipe_wall                   = 1

###############################################################################
# HELPERS
###############################################################################

def generate_hole_positions(holes_per_row, col_spacing, row_spacing):
    holes = []
    n_rows = len(holes_per_row)
    row_offsets = [(i - (n_rows - 1) / 2) * row_spacing for i in range(n_rows)]
    for row_idx, n_holes in enumerate(holes_per_row):
        y = row_offsets[row_idx]
        col_offsets = [(i - (n_holes - 1) / 2) * col_spacing for i in range(n_holes)]
        for x in col_offsets:
            holes.append((x, y))
    return holes

###############################################################################
# 1.  MAIN SHELL BODY
###############################################################################
outer_cyl = (
    cq.Workplane("front")
    .circle(R + shell_thickness)
    .extrude(H)
    .edges("front").fillet(fillet_r)
    .edges("back").fillet(fillet_r)
)

inner_bore = (
    cq.Workplane("front")
    .circle(R)
    .extrude(H)
    .edges("front").fillet(fillet_r)
    .edges("back").fillet(fillet_r)
)

# keep inner_bore_solid as reference for trimming side nozzles later
inner_bore_solid = inner_bore   # filleted R cylinder, used as cookie cutter
outer_cyl_solid  = outer_cyl    # filleted R+t cylinder, used as cookie cutter

shell_main = outer_cyl.cut(inner_bore)

###############################################################################
# 2.  AXIAL NOZZLES (inner fluid inlet/outlet)
###############################################################################
nozzle_bot_outer = (
    cq.Workplane("front")
    .workplane(offset=-inner_fluid_depth)
    .circle(inner_fluid_r + shell_thickness)
    .extrude(inner_fluid_depth)
)
nozzle_top_outer = (
    cq.Workplane("front")
    .workplane(offset=H)
    .circle(inner_fluid_r + shell_thickness)
    .extrude(inner_fluid_depth)
)

nozzle_bot_bore = (
    cq.Workplane("front")
    .workplane(offset=-inner_fluid_depth)
    .circle(inner_fluid_r)
    .extrude(inner_fluid_depth + fillet_r)
)
nozzle_top_bore = (
    cq.Workplane("front")
    .workplane(offset=H - fillet_r)
    .circle(inner_fluid_r)
    .extrude(inner_fluid_depth + fillet_r)
)

shell_with_axial_nozzles = (
    shell_main
    .union(nozzle_bot_outer)
    .union(nozzle_top_outer)
    .cut(nozzle_bot_bore)
    .cut(nozzle_top_bore)
)

###############################################################################
# 3.  SIDE NOZZLES — conform-and-connect approach
#
#  For each side nozzle:
#  a) make the fluid bore cylinder (full length, starting from axis y=0)
#  b) make the shell around it (annular tube wall, same full length)
#  c) trim the bore so it only exists OUTSIDE the main inner bore
#     (cut away the portion inside the main cylinder interior)
#  d) trim the shell wall so it only exists OUTSIDE the main outer cylinder
#     (cut away the portion inside the main shell wall)
#  e) the trimmed shell wall now has a curved face that sits flush on the
#     main cylinder outer surface — fuse it to shell_with_axial_nozzles
#  f) the trimmed bore connects flush at the inner shell surface
###############################################################################

# full length from axis to nozzle tip
full_length = R + shell_thickness + outer_fluid_depth

# --- INLET (+Y direction) ---

# step a: fluid bore, full length from axis
inlet_bore_full = (
    cq.Workplane("XZ")
    .transformed(offset=(0, outer_fluid_inlet_z, 0))
    .circle(outer_fluid_r)
    .extrude(full_length)
)

# step b: nozzle shell wall, full length from axis
inlet_shell_full = (
    cq.Workplane("XZ")
    .transformed(offset=(0, outer_fluid_inlet_z, 0))
    .circle(outer_fluid_r + shell_thickness)
    .extrude(full_length)
    .cut(inlet_bore_full)   # hollow it
)

# step c: trim bore — remove the part inside the main inner bore
#          (the part of the bore that's inside the main cylinder interior
#           belongs to fluid_2, not to the nozzle bore geometry)
inlet_bore_trimmed = inlet_bore_full.cut(inner_bore_solid)

# step d: trim shell wall — remove the part inside the main outer cylinder
#          (the nozzle wall only exists outside the main shell body)
inlet_shell_trimmed = inlet_shell_full.cut(outer_cyl_solid)

# --- OUTLET (-Y direction) ---

outlet_bore_full = (
    cq.Workplane("XZ")
    .transformed(offset=(0, outer_fluid_outlet_z, 0))
    .circle(outer_fluid_r)
    .extrude(-full_length)
)

outlet_shell_full = (
    cq.Workplane("XZ")
    .transformed(offset=(0, outer_fluid_outlet_z, 0))
    .circle(outer_fluid_r + shell_thickness)
    .extrude(-full_length)
    .cut(outlet_bore_full)
)

outlet_bore_trimmed = outlet_bore_full.cut(inner_bore_solid)
outlet_shell_trimmed = outlet_shell_full.cut(outer_cyl_solid)

# step e+f: fuse trimmed nozzle shells into main shell
solid_shell = (
    shell_with_axial_nozzles
    .union(inlet_shell_trimmed)
    .union(outlet_shell_trimmed)
)

###############################################################################
# 4.  TUBE BUNDLE
###############################################################################
holes = generate_hole_positions(holes_per_row, col_spacing, row_spacing)
dz    = baffle_end - baffle_start + baffle_thickness

tube_wall_parts  = []
tube_bore_parts  = []
tube_solid_parts = []

for (x, y) in holes:
    r_out = hole_diameter / 2
    r_in  = r_out - pipe_wall

    tube_solid = (
        cq.Workplane("front")
        .workplane(offset=baffle_start)
        .center(x, y)
        .circle(r_out)
        .extrude(dz)
    )
    tube_bore = (
        cq.Workplane("front")
        .workplane(offset=baffle_start)
        .center(x, y)
        .circle(r_in)
        .extrude(dz)
    )
    tube_wall_parts.append(tube_solid.cut(tube_bore))
    tube_bore_parts.append(tube_bore)
    tube_solid_parts.append(tube_solid)

solid_pipes = tube_wall_parts[0]
for t in tube_wall_parts[1:]:
    solid_pipes = solid_pipes.union(t)

pipes_for_cut = tube_solid_parts[0]
for t in tube_solid_parts[1:]:
    pipes_for_cut = pipes_for_cut.union(t)

###############################################################################
# 5.  BAFFLES
###############################################################################
def make_baffle_disc(z_pos):
    return (
        cq.Workplane("front")
        .workplane(offset=z_pos)
        .circle(baffle_radius)
        .extrude(baffle_thickness)
        .cut(pipes_for_cut)
    )

def make_semi_baffle(index, z_pos):
    disc = (
        cq.Workplane("front")
        .workplane(offset=z_pos)
        .circle(baffle_radius)
        .extrude(baffle_thickness)
    )
    if index % 2 == 0:
        cut_box = (
            cq.Workplane("front")
            .workplane(offset=z_pos)
            .rect(2 * baffle_radius, 2 * baffle_radius)
            .extrude(baffle_thickness)
            .translate((0, baffle_radius / 2, 0))
        )
    else:
        cut_box = (
            cq.Workplane("front")
            .workplane(offset=z_pos)
            .rect(2 * baffle_radius, 2 * baffle_radius)
            .extrude(baffle_thickness)
            .translate((0, -baffle_radius / 2, 0))
        )
    return disc.intersect(cut_box).cut(pipes_for_cut)

baffle1 = make_baffle_disc(baffle_start)
baffle2 = make_baffle_disc(baffle_end)

semi_spacing = (baffle_end - baffle_start) / (num_semi_baffles + 1)
semi_list = [
    make_semi_baffle(i, baffle_start + (i + 1) * semi_spacing)
    for i in range(num_semi_baffles)
]

solid_baffles = baffle1.union(baffle2)
for sb in semi_list:
    solid_baffles = solid_baffles.union(sb)

###############################################################################
# 6.  FLUID 1 (tube side)
###############################################################################
interior = inner_bore_solid

plenum_bot = interior.intersect(
    cq.Workplane("front")
    .workplane(offset=-inner_fluid_depth)
    .rect(2 * R, 2 * R)
    .extrude(inner_fluid_depth + baffle_start)
)

plenum_top = interior.intersect(
    cq.Workplane("front")
    .workplane(offset=baffle_end + baffle_thickness)
    .rect(2 * R, 2 * R)
    .extrude(H - (baffle_end + baffle_thickness) + inner_fluid_depth)
)

fluid1_bore_bot = (
    cq.Workplane("front")
    .workplane(offset=-inner_fluid_depth)
    .circle(inner_fluid_r)
    .extrude(inner_fluid_depth + fillet_r)
)
fluid1_bore_top = (
    cq.Workplane("front")
    .workplane(offset=H - fillet_r)
    .circle(inner_fluid_r)
    .extrude(inner_fluid_depth + fillet_r)
)

fluid1_tubes = tube_bore_parts[0]
for t in tube_bore_parts[1:]:
    fluid1_tubes = fluid1_tubes.union(t)

fluid_1 = (
    plenum_bot
    .union(plenum_top)
    .union(fluid1_tubes)
    .union(fluid1_bore_bot)
    .union(fluid1_bore_top)
)

###############################################################################
# 7.  FLUID 2 (shell side)
#     annular space + trimmed side nozzle bores
#     trimmed bores connect flush at inner shell surface with no overshoot
###############################################################################
fluid_2 = (
    interior
    .cut(plenum_bot)
    .cut(plenum_top)
    .cut(solid_baffles)
    .cut(pipes_for_cut)
    .union(inlet_bore_trimmed)
    .union(outlet_bore_trimmed)
)

solid_shell = solid_shell.cut(fluid_2)

###############################################################################
# 8.  ASSEMBLY
###############################################################################
hx = cq.Assembly()
hx.add(solid_shell,   name="solid_shell",   color=cq.Color("gray"))
hx.add(solid_pipes,   name="solid_pipes",   color=cq.Color("blue"))
hx.add(solid_baffles, name="solid_baffles", color=cq.Color("red"))
hx.add(fluid_1,       name="fluid_1",       color=cq.Color("cyan",  alpha=0.4))
hx.add(fluid_2,       name="fluid_2",       color=cq.Color("green", alpha=0.4))

hx.save("hx_with_thickness.step")

+ccc+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [27]:
full_geometry = cq.Compound.makeCompound([
    solid_shell.val(),
    solid_pipes.val(),
    solid_baffles.val(),
    fluid_1.val(),
    fluid_2.val()
])

cq.exporters.export(full_geometry, "hx_w_thickness.brep")